# Guía de Estudio QKP - Cuaderno 1: Teoría del Problema de la Mochila Cuadrática (QKP) y Modelado Manual

Bienvenido a tu spin-off de autoaprendizaje sobre el **Quadratic Knapsack Problem (QKP)**. Este cuaderno está diseñado para que estudies a fondo durante una semana la transición del modelo lineal al cuadrático, haciendo los cálculos a mano antes de programar.

## 1. ¿Qué es el QKP y por qué es más difícil?

En el Problema de la Mochila clásico (Lineal), cada objeto $i$ tiene un valor individual $v_i$ y un peso $w_i$. El objetivo es seleccionar un subconjunto de objetos para maximizar el valor total sin superar la capacidad $W$:
$$\max \sum_{i} v_i x_i \quad \text{s.t.} \quad \sum_{i} w_i x_i \le W$$

En el **QKP**, añadimos **sinergias o interacciones cuadráticas** entre pares de objetos. Si seleccionas el objeto $i$ y el objeto $j$ juntos, obtienes un valor adicional (o una penalización) $v_{ij}$. La formulación matemática es:

$$\max \sum_{i=1}^{n} v_i x_i + \sum_{i < j} v_{ij} x_i x_j$$
$$\text{sujeto a:} \quad \sum_{i=1}^{n} w_i x_i \le W$$
$$x_i \in \{0, 1\}$$

### ¿Por qué se rompe la computación clásica?
1. **Pérdida de la Programación Dinámica:** En el caso lineal, podemos resolver el problema óptimamente con la ecuación de Bellman en tiempo $O(N \cdot W)$. En QKP, el valor de meter un objeto depende de qué *otros* objetos ya estén en la mochila, destruyendo la subestructura óptima requerida por la programación dinámica.
2. **NP-Hard Estricto:** QKP es NP-hard en sentido fuerte. Los solvers clásicos (como PuLP o Gurobi) tienen que linealizar el problema introduciendo variables adicionales $z_{ij} = x_i x_j$, lo que incrementa el número de variables de $N$ a $N + \frac{N(N-1)}{2}$. Para $N = 100$, ¡esto significa pasar de 100 variables a más de 5,000 variables y restricciones de linealización!

## 2. Instancia de Demostración (4 objetos)

Modelaremos a mano el siguiente problema pequeño:
* **Objetos:** $n = 4$
* **Capacidad:** $W = 5$ kg

### Datos de los Objetos:
| Objeto | Peso ($w_i$) | Valor Individual ($v_i$) |
| :---: | :---: | :---: |
| 1 | 2 | 5 |
| 2 | 3 | 6 |
| 3 | 4 | 8 |
| 4 | 2 | 4 |

### Matriz de Sinergias ($v_{ij}$):
* $v_{12} = 3$ (Bono si llevamos Objeto 1 y Objeto 2 juntos)
* $v_{23} = -2$ (Castigo si llevamos Objeto 2 y Objeto 3 juntos - incompatibilidad)
* $v_{14} = 4$ (Bono si llevamos Objeto 1 y Objeto 4 juntos)
* Todas las demás sinergias $v_{ij} = 0$.

### Función Objetivo a Maximizar:
$$f(x) = 5x_1 + 6x_2 + 8x_3 + 4x_4 + 3x_1 x_2 - 2x_2 x_3 + 4x_1 x_4$$

## 3. Desarrollo del Modelo QUBO por Pasos (A mano)

### Paso A: Determinar y Expandir la Variable de Holgura (Slack)
La restricción es de desigualdad:
$$2x_1 + 3x_2 + 4x_3 + 2x_4 \le 5$$

La convertimos en igualdad sumando la holgura $s$:
$$2x_1 + 3x_2 + 4x_3 + 2x_4 + s = 5$$

**Rango de $s$:**
* Si la mochila está vacía ($x = 0$), $s = 5$.
* Si la mochila está llena, $s = 0$.
* Rango de $s$: $\{0, 1, 2, 3, 4, 5\}$ (máximo 5).

**Expansión Binaria:**
Para cubrir hasta el valor 5, necesitamos $M = \lceil \log_2(5+1) \rceil = 3$ bits ($s_0, s_1, s_2$):
$$s = 1 \cdot s_0 + 2 \cdot s_1 + 4 \cdot s_2$$
*Nota: Aunque 3 bits pueden sumar hasta 7 ($1+2+4$), el optimizador nunca elegirá $s > 5$ porque aumentaría la penalización de forma innecesaria.*

### Paso B: Construir el Término de Penalización Cuadrática
Usamos una fuerza de penalización $\lambda$. Para asegurar que la restricción sea inviolable, una buena regla práctica es elegir $\lambda$ mayor que el máximo valor posible del problema. Aquí elegiremos **$\lambda = 10$**.

La penalización es:
$$P(x, s) = \lambda \left(2x_1 + 3x_2 + 4x_3 + 2x_4 + s_0 + 2s_1 + 4s_2 - 5\right)^2$$

Vamos a expandir algebraicamente el término al cuadrado:
$$(\sum a_i y_i - W)^2 = \sum a_i(a_i - 2W)y_i + 2\sum_{i < j} a_i a_j y_i y_j + W^2$$
Donde $y = (x_1, x_2, x_3, x_4, s_0, s_1, s_2)$ con coeficientes $a = (2, 3, 4, 2, 1, 2, 4)$ y $W = 5$.

### Paso C: Cálculo de Coeficientes de la Penalización ($P(y)$)

**Términos Diagonales de la Penalización ($P_{ii}$):**
Fórmula: $P_{ii} = \lambda \cdot a_i(a_i - 2W) = 10 \cdot a_i(a_i - 10)$:
* $x_1$: $10 \cdot 2(2 - 10) = 10 \cdot 2(-8) = -160$
* $x_2$: $10 \cdot 3(3 - 10) = 10 \cdot 3(-7) = -210$
* $x_3$: $10 \cdot 4(4 - 10) = 10 \cdot 4(-6) = -240$
* $x_4$: $10 \cdot 2(2 - 10) = 10 \cdot 2(-8) = -160$
* $s_0$: $10 \cdot 1(1 - 10) = 10 \cdot 1(-9) = -90$
* $s_1$: $10 \cdot 2(2 - 10) = 10 \cdot 2(-8) = -160$
* $s_2$: $10 \cdot 4(4 - 10) = 10 \cdot 4(-6) = -240$

**Términos Cruzados de la Penalización ($P_{ij}$ para $i < j$):**
Fórmula: $P_{ij} = 2 \lambda \cdot a_i a_j = 20 \cdot a_i a_j$:
* $x_1 x_2$: $20 \cdot (2 \cdot 3) = 120$
* $x_1 x_3$: $20 \cdot (2 \cdot 4) = 160$
* $x_1 x_4$: $20 \cdot (2 \cdot 2) = 80$
* $x_2 x_3$: $20 \cdot (3 \cdot 4) = 240$
* $x_2 x_4$: $20 \cdot (3 \cdot 2) = 120$
* $x_3 x_4$: $20 \cdot (4 \cdot 2) = 160$
* Interacciones con Slack:
  * $x_1 s_0$: $20 \cdot (2 \cdot 1) = 40$
  * $x_1 s_1$: $20 \cdot (2 \cdot 2) = 80$
  * $x_1 s_2$: $20 \cdot (2 \cdot 4) = 160$
  * (y así sucesivamente para todas las combinaciones $y_i y_j$).

### Paso D: Juntar la Penalización con el Objetivo
En QUBO queremos **minimizar** la energía. Como el objetivo original se quería **maximizar**, lo restamos (multiplicamos por $-1$):
$$H_{QUBO} = - f(x) + P(x, s)$$

**Matriz QUBO Final $Q$:**
* **Diagonales $Q_{ii}$:** $P_{ii} - v_i$
  * $Q_{11} = -160 - 5 = -165$
  * $Q_{22} = -210 - 6 = -216$
  * $Q_{33} = -240 - 8 = -248$
  * $Q_{44} = -160 - 4 = -164$
  * $Q_{s_0s_0} = -90 - 0 = -90$
  * $Q_{s_1s_1} = -160 - 0 = -160$
  * $Q_{s_2s_2} = -240 - 0 = -248$

* **Fuera de la diagonal $Q_{ij}$ ($i < j$):** $P_{ij} - v_{ij}$
  * $Q_{12} = 120 - 3 = 117$ (El bono de valor de 3 reduce la penalización de peso)
  * $Q_{23} = 240 - (-2) = 242$ (La incompatibilidad de -2 aumenta el costo)
  * $Q_{14} = 80 - 4 = 76$
  * Todas las demás interacciones $Q_{ij}$ son exactamente iguales a $P_{ij}$ porque no tienen sinergia asociada ($v_{ij}=0$).

## 4. Ejercicios Prácticos para tu Semana de Estudio

1. **Verificación Manual:** Calcula a mano el valor de la energía QUBO $y^T Q y$ para la combinación factible $x = (1, 1, 0, 0)$ (Laptop + Cámara). Su peso es $2+3=5$ kg (justo al límite), por lo que $s=0$ (`000` en bits de holgura).
2. **Verificación No Factible:** Calcula la energía para la combinación $x = (1, 1, 1, 0)$ (Laptop + Cámara + Proyector). Su peso es $2+3+4=9$ kg. Encuentra qué valor de $s$ minimiza la penalización y comprueba que la energía QUBO final es mucho más alta que en el caso factible.

En el siguiente cuaderno, programaremos esta instancia en Python clásico y analizaremos cómo escala el problema combinatorio a tamaños más grandes.